In [ ]:
import pickle
import os
import re 
import json 
import click
import torch
import dnnlib.util_v4 as util
from torch_utils import distributed as dist
from training import toy_training_loop_vfm_noDataGen
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


import warnings
warnings.filterwarnings('ignore', 'Grad strides do not match bucket view strides') # False warning 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [ ]:
def plotTraj(data, data_sim = None, title = None):
    data_min = np.min(data)
    data_max = np.max(data)
    data_range = data_max - data_min
    offset = data_range * 1.1

    time = np.arange(data.shape[0])
    if data_sim is not None:
        time_sim = np.arange(data_sim.shape[0])
    
    # Plot the trajectories
    plt.figure(figsize=(6, 4))  # Adjust figure size as needed
    for i in range(data.shape[1]):
        adjusted_y = data[:, i] + i * offset
        plt.plot(time, adjusted_y, label=f'Trajectory {i+1}', linewidth=0.5, c='red') 
        if data_sim is not None:
            adjusted_y_sim = data_sim[:, i] + i * offset
            plt.plot(time_sim, adjusted_y_sim, label=f'Trajectory {i+1}', linewidth=0.5, c='black')
        
    # Customize the plot
    plt.xlabel('Time')
    plt.tight_layout()
    if title is not None:
        plt.title(title)
    plt.show()

In [ ]:
dt = 0.001
T_min = 999

# 1. no covariate (time), no x0_tau

In [ ]:
folder = 'out/test/00010-hist_tau1-prr-uncond-regularFM-gpus1-batch256-fp32-with_covariates'
pkl_path = folder + '/network-snapshot-010035.pkl'

folder_data = 'data/toy_test'
dataset_samples = np.load(folder_data + "/dataset_samples.npz", allow_pickle=True)

In [ ]:
data_raw = dataset_samples['samples']
dset_samples_raw = []
for ii in range(len(data_raw)):
    dset_samples_raw.append(np.array(data_raw[ii]))

In [ ]:
with open(pkl_path, 'rb') as f:
    model_all = pickle.load(f)

if 'ema' in model_all:
    net = model_all['ema']  # EMA-stabilized model (recommended for inference/simulation)
else:
    net = model_all['net']  # Raw trained model

net = net.to(device).eval()
flow_net = net.unet_model  # Compressive flow (u_theta)
dyn_net = net.vnet_model   # Dynamics flow (v_theta)
encoder = net.encoder      # For latent proposals

In [ ]:
lag = 1
cov_static_samples = []
for ii in range(len(dset_samples_raw)):
    lagged_samples = [dset_samples_raw[ii][j : (j - lag if j - lag != 0 else None), :] for j in range(lag, -1, -1)]
    result = np.concatenate(lagged_samples, axis=-1)
    cov_static_samples.append(result)

In [ ]:
dset_samples = []
for ii in range(len(dset_samples_raw)):
    dset_samples.append(dset_samples_raw[ii][lag:,:])

In [ ]:
data_raw = dset_samples.copy()

In [ ]:
starting_pts = np.zeros((len(data_raw), data_raw[0].shape[1]))
for ii in range(len(data_raw)):
    starting_pts[ii,:] = data_raw[ii][0,:]
starting_pts = torch.from_numpy(starting_pts).unsqueeze(1).type(torch.float32).to(device) #ntrajs,1,dim

curr_tau_pts = starting_pts
# covariates_static = torch.stack([torch.tensor(cov_static_samples[jj][0],
#                                                           dtype=torch.float32) 
#                                             for jj in range(len(cov_static_samples))]).to(curr_tau_pts.device)

covariates_static = torch.stack([
    torch.tensor(cov_static_samples[j][0], dtype=torch.float32)
    for j in range(len(cov_static_samples))
], dim=0).to(curr_tau_pts.device)

In [ ]:
n_step = 999
tau = 1.0
curr_tau_trajs = []
curr_tau_trajs.append(curr_tau_pts.cpu().numpy())
include_x0_tau = False

In [ ]:
for ii in tqdm(range(n_step-1)):
    curr_tau_pts = util.calc_dyn_trajectories(dyn_net, curr_tau_pts.squeeze(1),
                                               tau, x0_tau=None, 
                                               covariates=None,
                                               next_covariates=None,
                                               covariates_static=covariates_static,
                                               include_x0_tau=include_x0_tau, nt=2)
    curr_tau_pts = curr_tau_pts[-1].unsqueeze(1)
    curr_tau_trajs.append(curr_tau_pts.cpu().numpy())

    lagged_trajs = [curr_tau_trajs[-1 - j] for j in range(0, lag + 1)]
    covariates_static = torch.tensor(np.concatenate(lagged_trajs, axis=-1).squeeze(1), dtype=torch.float32)
    
curr_tau_trajs = np.concatenate(curr_tau_trajs, axis=1)

In [ ]:
trial_plot = 19
plotTraj(data_raw[trial_plot], data_sim = curr_tau_trajs[trial_plot,:,:], title = 'trial_' + str(trial_plot))

In [ ]:
## appendix...
# oracle checking...

starting_pts = np.zeros((len(data_raw), data_raw[0].shape[1]))
for ii in range(len(data_raw)):
    starting_pts[ii,:] = data_raw[ii][0,:]
starting_pts = torch.from_numpy(starting_pts).unsqueeze(1).type(torch.float32).to(device) #ntrajs,1,dim

cov_dynamic_samples = None

curr_tau_pts = starting_pts
n_step = 999
tau = 1.0
curr_tau_trajs = []
curr_tau_trajs.append(curr_tau_pts.cpu().numpy())
include_x0_tau = False

for ii in tqdm(range(n_step-1)):

    covariates_static = torch.stack([
        torch.tensor(cov_static_samples[j][ii], dtype=torch.float32)
        for j in range(len(cov_static_samples))
    ]).to(device)

    x0_ora = torch.stack([
        torch.tensor(data_raw[j][ii], dtype=torch.float32)
        for j in range(len(data_raw))
    ]).to(device)

    
    curr_tau_pts = util.calc_dyn_trajectories(dyn_net, x0_ora,
                                               tau, x0_tau=None, 
                                               covariates=None,
                                               next_covariates=None,
                                               covariates_static=covariates_static,
                                               include_x0_tau=include_x0_tau, nt=2)
    curr_tau_pts = curr_tau_pts[-1].unsqueeze(1)
    curr_tau_trajs.append(curr_tau_pts.cpu().numpy())
   
curr_tau_trajs = np.concatenate(curr_tau_trajs, axis=1)

In [ ]:
trial_plot = 3
plotTraj(data_raw[trial_plot], data_sim = curr_tau_trajs[trial_plot,:,:], title = 'trial_' + str(trial_plot))